# Experimento VCP adaptado a FX: EURUSD Diario vs Horario

Parametros adaptados a la dinamica de monedas:
- Contracciones mas chicas (max_depth_pct 0.10, max_depth_atr 3)
- Stops mas apretados (2% stop, ATR trailing 1.5-2x)
- Target exit en R-multiplos
- Sin trend template (FX no tiene Etapa 2 sostenida)

In [ ]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.configs import ATRZigZagConfig
from vcp_detection.heuristic import ATRZigZagDetector, run_full_vcp_pipeline
from vcp_detection.analysis import (
    group_signals_into_patterns,
    simulate_trade,
    plot_vcp_pattern,
    plot_trade_simulation,
)

pd.set_option("display.float_format", "{:.4f}".format)
print("Imports OK")

## 1. Configuracion FX

Dos sets de parametros: uno para acciones (referencia) y uno adaptado a FX.

In [ ]:
# === PARAMETROS ACCIONES (referencia, los que usamos en el experimento principal) ===
STOCK_PARAMS = {
    "swing": ATRZigZagConfig(atr_length=14, atr_mult=2.0, use_close_only=False),
    "sequence": {
        "method": "tolerance", "min_contractions": 2, "max_contractions": 6,
        "lookback_bars": 126, "tolerance": 0.10,
        "max_depth_pct": 0.35, "max_depth_atr": 7, "min_total_reduction": 0.80,
        "max_gap_between_contractions_days": None,
        "require_ascending_lows": True, "ascending_lows_tolerance": 0.10,
    },
    "compression": {"method": "ratio", "atr_period": 14, "ratio_threshold": 0.85},
    "volume_contraction": {"method": "ratio", "volume_column": "volume", "ratio_threshold": 0.85},
    "breakout": {
        "volume_method": "ratio", "volume_ratio_threshold": 1.5,
        "volume_lookback_days": 50, "require_volume_confirmation": False,
        "max_entry_distance_pct": 0.10,
    },
    "risk": {
        "max_stop_loss_pct": 0.05, "breakeven_r_multiple": 2.0,
        "trailing_sma_period": 20, "trailing_volume_factor": 1.5,
        "trailing_stop_method": "atr", "trailing_atr_period": 14,
        "trailing_atr_multiplier": 3.0,
        "max_bars_without_progress": 20, "min_progress_r": 0.5,
        "early_exit_days": 3, "target_r_multiple": None,
    },
}

# === PARAMETROS FX (adaptados a monedas) ===
FX_PARAMS = {
    "swing": ATRZigZagConfig(atr_length=14, atr_mult=1.5, use_close_only=False),
    "sequence": {
        "method": "tolerance", "min_contractions": 2, "max_contractions": 6,
        "lookback_bars": 126, "tolerance": 0.10,
        "max_depth_pct": 0.10,       # FX: contracciones max 10% (vs 35% acciones)
        "max_depth_atr": 3,           # FX: max 3 ATR (vs 7 acciones)
        "min_total_reduction": 0.60,  # FX: relajar compresion minima (vs 0.80)
        "max_gap_between_contractions_days": None,
        "require_ascending_lows": True,
        "ascending_lows_tolerance": 0.03,  # FX: mas estricto (vs 0.10)
    },
    "compression": {"method": "ratio", "atr_period": 14, "ratio_threshold": 0.85},
    "volume_contraction": {"method": "ratio", "volume_column": "volume", "ratio_threshold": 0.85},
    "breakout": {
        "volume_method": "ratio", "volume_ratio_threshold": 1.5,
        "volume_lookback_days": 50,
        "require_volume_confirmation": False,  # FX: sin vol confirmation
        "max_entry_distance_pct": 0.03,        # FX: 3% max (vs 10%)
    },
    "risk": {
        "max_stop_loss_pct": 0.02,          # FX: 2% stop (vs 5%)
        "breakeven_r_multiple": 1.0,        # FX: breakeven rapido (vs 2.0)
        "trailing_sma_period": 20,
        "trailing_volume_factor": 1.5,
        "trailing_stop_method": "atr",
        "trailing_atr_period": 14,
        "trailing_atr_multiplier": 2.0,    # FX: trailing mas apretado (vs 3.0)
        "max_bars_without_progress": 15,   # FX: time exit mas agresivo
        "min_progress_r": 0.5,
        "early_exit_days": 3,
        "target_r_multiple": 3.0,          # FX: take profit a 3R
    },
}

print("Diferencias clave FX vs Acciones:")
diffs = [
    ("max_depth_pct", STOCK_PARAMS["sequence"]["max_depth_pct"], FX_PARAMS["sequence"]["max_depth_pct"]),
    ("max_depth_atr", STOCK_PARAMS["sequence"]["max_depth_atr"], FX_PARAMS["sequence"]["max_depth_atr"]),
    ("min_total_reduction", STOCK_PARAMS["sequence"]["min_total_reduction"], FX_PARAMS["sequence"]["min_total_reduction"]),
    ("ascending_lows_tol", STOCK_PARAMS["sequence"]["ascending_lows_tolerance"], FX_PARAMS["sequence"]["ascending_lows_tolerance"]),
    ("max_entry_distance", STOCK_PARAMS["breakout"]["max_entry_distance_pct"], FX_PARAMS["breakout"]["max_entry_distance_pct"]),
    ("max_stop_loss", STOCK_PARAMS["risk"]["max_stop_loss_pct"], FX_PARAMS["risk"]["max_stop_loss_pct"]),
    ("breakeven_r", STOCK_PARAMS["risk"]["breakeven_r_multiple"], FX_PARAMS["risk"]["breakeven_r_multiple"]),
    ("trailing_atr_mult", STOCK_PARAMS["risk"]["trailing_atr_multiplier"], FX_PARAMS["risk"]["trailing_atr_multiplier"]),
    ("target_r", STOCK_PARAMS["risk"]["target_r_multiple"], FX_PARAMS["risk"]["target_r_multiple"]),
    ("atr_mult (swing)", STOCK_PARAMS["swing"].atr_mult, FX_PARAMS["swing"].atr_mult),
]
for name, stock, fx in diffs:
    print(f"  {name:<22} Acciones={stock}  ->  FX={fx}")

## 2. Cargar datos EURUSD

In [ ]:
daily = pd.read_csv(
    project_root / "data" / "monedas" / "EURUSD.csv",
    parse_dates=["date"], index_col="date",
)
hourly = pd.read_csv(
    project_root / "data" / "monedas_hora" / "EURUSD.csv",
    parse_dates=["date"], index_col="date",
)

print(f"EURUSD diario:  {len(daily):,} barras ({daily.index.min().date()} a {daily.index.max().date()})")
print(f"EURUSD horario: {len(hourly):,} barras ({hourly.index.min()} a {hourly.index.max()})")

## 3. Funcion de analisis

In [ ]:
def run_analysis(ohlc, params, use_vol_contraction=False, label=""):
    """Corre el pipeline VCP y retorna metricas + trades."""
    detector = ATRZigZagDetector(params["swing"])
    vol_params = params["volume_contraction"] if use_vol_contraction else None

    results = run_full_vcp_pipeline(
        ohlc=ohlc, swing_detector=detector,
        sequence_params=params["sequence"],
        compression_params=params["compression"],
        breakout_params=params["breakout"],
        volume_contraction_params=vol_params,
    )

    signals = {dt: s for dt, s in results.items() if s is not None}
    patterns = group_signals_into_patterns(signals, risk_params=params["risk"])

    trades = []
    for p in patterns:
        t = simulate_trade(ohlc, p, params["risk"])
        t["pattern"] = p
        trades.append(t)

    n = len(trades)
    if n > 0:
        wins = sum(1 for t in trades if t["pnl_pct"] > 0)
        cr = float(np.prod([1 + t["pnl_pct"] for t in trades]) - 1)
        avg_r = float(np.mean([t["r_multiple"] for t in trades]))
        avg_pnl = float(np.mean([t["pnl_pct"] for t in trades]))
        max_r = float(max(t["max_r"] for t in trades))
    else:
        wins, cr, avg_r, avg_pnl, max_r = 0, 0, 0, 0, 0

    # Exit reason breakdown
    reasons = {}
    for t in trades:
        r = t["exit_reason"]
        reasons[r] = reasons.get(r, 0) + 1

    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(f"  Senales: {len(signals)}, Patrones: {len(patterns)}, Trades: {n}")
    if n > 0:
        print(f"  Win/Loss: {wins}W / {n-wins}L  (WR: {wins/n:.0%})")
        print(f"  Retorno acum: {cr:+.2%}")
        print(f"  Avg PnL/trade: {avg_pnl:+.3%}")
        print(f"  Avg R: {avg_r:+.2f}R, Max R alcanzado: {max_r:.1f}R")
        print(f"  Salidas: {reasons}")

    return {"signals": signals, "patterns": patterns, "trades": trades,
            "n_trades": n, "wins": wins, "cr": cr, "avg_r": avg_r, "reasons": reasons}

print("Funcion definida.")

## 4. Experimento: Params Acciones vs Params FX

In [ ]:
results = {}

for data_label, ohlc in [("Diario", daily), ("Horario", hourly)]:
    for param_label, params in [("Stock", STOCK_PARAMS), ("FX", FX_PARAMS)]:
        key = f"{data_label}_{param_label}"
        results[key] = run_analysis(ohlc, params, label=f"EURUSD {data_label} - Params {param_label}")

In [ ]:
# Tabla resumen
rows = []
for key, r in results.items():
    data, params = key.split("_")
    n = r["n_trades"]
    rows.append({
        "Data": data,
        "Params": params,
        "Senales": len(r["signals"]),
        "Patrones": len(r["patterns"]),
        "Trades": n,
        "Wins": r["wins"],
        "Losses": n - r["wins"],
        "WR": r["wins"] / n if n > 0 else 0,
        "CR": r["cr"],
        "Avg R": r["avg_r"],
    })

summary = pd.DataFrame(rows)
display(summary.style.format({
    "WR": "{:.0%}", "CR": "{:+.2%}", "Avg R": "{:+.2f}",
}).background_gradient(subset=["CR"], cmap="RdYlGn", vmin=-0.1, vmax=0.1))

print("\nExit reasons por config:")
for key, r in results.items():
    print(f"  {key}: {r['reasons']}")

## 5. Plots de trades (mejor config)

In [ ]:
# Elegir la config con mas trades para plotear
best_key = max(results, key=lambda k: results[k]["n_trades"])
best = results[best_key]
best_data = daily if "Diario" in best_key else hourly
best_params = STOCK_PARAMS if "Stock" in best_key else FX_PARAMS

print(f"Ploteando: {best_key} ({best['n_trades']} trades)")

for i, (pat, trade) in enumerate(zip(best["patterns"], best["trades"]), 1):
    if i > 10:
        print(f"... y {len(best['trades']) - 10} trades mas")
        break
    pnl = trade["pnl_pct"]
    r = trade["r_multiple"]
    print(f"\nTrade {i}: {trade['exit_reason']} | PnL={pnl:+.2%} | R={r:+.1f}R")
    plot_trade_simulation(best_data, pat, trade, pattern_number=i,
                          risk_params=best_params["risk"], ticker="EURUSD")
    plt.show()

## 6. Grilla de target R y trailing ATR mult (params FX)

In [ ]:
TARGET_RS = [None, 2.0, 3.0, 5.0]
ATR_MULTS = [1.5, 2.0, 2.5]

grid_results = []

for data_label, ohlc in [("Diario", daily), ("Horario", hourly)]:
    for target_r in TARGET_RS:
        for atr_m in ATR_MULTS:
            params = {
                **FX_PARAMS,
                "risk": {
                    **FX_PARAMS["risk"],
                    "target_r_multiple": target_r,
                    "trailing_atr_multiplier": atr_m,
                },
            }
            detector = ATRZigZagDetector(params["swing"])
            res = run_full_vcp_pipeline(
                ohlc=ohlc, swing_detector=detector,
                sequence_params=params["sequence"],
                compression_params=params["compression"],
                breakout_params=params["breakout"],
                volume_contraction_params=None,
            )
            signals = {dt: s for dt, s in res.items() if s is not None}
            patterns = group_signals_into_patterns(signals, risk_params=params["risk"])
            trades = []
            for p in patterns:
                t = simulate_trade(ohlc, p, params["risk"])
                t["pattern"] = p
                trades.append(t)

            n = len(trades)
            wins = sum(1 for t in trades if t["pnl_pct"] > 0) if n else 0
            cr = float(np.prod([1 + t["pnl_pct"] for t in trades]) - 1) if n else 0
            avg_r = float(np.mean([t["r_multiple"] for t in trades])) if n else 0
            reasons = {}
            for t in trades:
                reasons[t["exit_reason"]] = reasons.get(t["exit_reason"], 0) + 1

            grid_results.append({
                "data": data_label,
                "target_R": str(target_r) if target_r else "None",
                "atr_mult": atr_m,
                "trades": n,
                "wins": wins,
                "WR": wins / n if n > 0 else 0,
                "CR": cr,
                "avg_R": avg_r,
                "targets": reasons.get("target", 0),
                "stops": reasons.get("stop_loss", 0) + reasons.get("trailing_stop", 0),
                "time_exits": reasons.get("time_exit", 0),
            })

grid_df = pd.DataFrame(grid_results)

for data_label in ["Diario", "Horario"]:
    print(f"\n{'='*60}")
    print(f"  EURUSD {data_label} - Grilla Target R x ATR Multiplier")
    print(f"{'='*60}")
    sub = grid_df[grid_df["data"] == data_label].drop(columns=["data"])
    display(sub.style.format({
        "WR": "{:.0%}", "CR": "{:+.2%}", "avg_R": "{:+.2f}",
    }).background_gradient(subset=["CR"], cmap="RdYlGn", vmin=-0.1, vmax=0.1))

## 7. Grilla de deteccion: depth y reduction (params FX)

In [ ]:
DEPTH_PCTS = [0.05, 0.08, 0.10, 0.15]
DEPTH_ATRS = [2, 3, 5]
REDUCTIONS = [0.50, 0.60, 0.70]

det_results = []

for data_label, ohlc in [("Diario", daily), ("Horario", hourly)]:
    for depth_pct in DEPTH_PCTS:
        for depth_atr in DEPTH_ATRS:
            for reduction in REDUCTIONS:
                params = {
                    **FX_PARAMS,
                    "sequence": {
                        **FX_PARAMS["sequence"],
                        "max_depth_pct": depth_pct,
                        "max_depth_atr": depth_atr,
                        "min_total_reduction": reduction,
                    },
                }
                detector = ATRZigZagDetector(params["swing"])
                res = run_full_vcp_pipeline(
                    ohlc=ohlc, swing_detector=detector,
                    sequence_params=params["sequence"],
                    compression_params=params["compression"],
                    breakout_params=params["breakout"],
                    volume_contraction_params=None,
                )
                signals = {dt: s for dt, s in res.items() if s is not None}
                patterns = group_signals_into_patterns(signals, risk_params=params["risk"])
                trades = []
                for p in patterns:
                    t = simulate_trade(ohlc, p, params["risk"])
                    t["pattern"] = p
                    trades.append(t)

                n = len(trades)
                wins = sum(1 for t in trades if t["pnl_pct"] > 0) if n else 0
                cr = float(np.prod([1 + t["pnl_pct"] for t in trades]) - 1) if n else 0

                det_results.append({
                    "data": data_label,
                    "depth_pct": depth_pct,
                    "depth_atr": depth_atr,
                    "reduction": reduction,
                    "signals": len(signals),
                    "patterns": len(patterns),
                    "trades": n,
                    "wins": wins,
                    "WR": wins / n if n > 0 else 0,
                    "CR": cr,
                })

det_df = pd.DataFrame(det_results)

for data_label in ["Diario", "Horario"]:
    print(f"\n{'='*60}")
    print(f"  EURUSD {data_label} - Grilla Deteccion")
    print(f"{'='*60}")
    sub = det_df[det_df["data"] == data_label].drop(columns=["data"])
    sub = sub.sort_values("trades", ascending=False)
    display(sub.style.format({
        "WR": "{:.0%}", "CR": "{:+.2%}",
    }).background_gradient(subset=["trades"], cmap="Blues")
    .background_gradient(subset=["CR"], cmap="RdYlGn", vmin=-0.1, vmax=0.1))

## 8. Resumen

In [ ]:
print("RESUMEN DEL EXPERIMENTO FX")
print("=" * 60)
print()
print("Comparacion Stock params vs FX params:")
display(summary.style.format({
    "WR": "{:.0%}", "CR": "{:+.2%}", "Avg R": "{:+.2f}",
}))

print("\nMejores configs de la grilla de salida (por CR):")
for data_label in ["Diario", "Horario"]:
    sub = grid_df[grid_df["data"] == data_label].sort_values("CR", ascending=False).head(3)
    print(f"\n  {data_label}:")
    for _, row in sub.iterrows():
        print(f"    target_R={row['target_R']}, atr_mult={row['atr_mult']} -> "
              f"{row['trades']} trades, WR={row['WR']:.0%}, CR={row['CR']:+.2%}")

print("\nMejores configs de la grilla de deteccion (por trades con CR>0):")
for data_label in ["Diario", "Horario"]:
    sub = det_df[(det_df["data"] == data_label) & (det_df["CR"] > 0)].sort_values("CR", ascending=False).head(3)
    print(f"\n  {data_label}:")
    for _, row in sub.iterrows():
        print(f"    depth_pct={row['depth_pct']}, depth_atr={row['depth_atr']}, red={row['reduction']} -> "
              f"{int(row['trades'])} trades, WR={row['WR']:.0%}, CR={row['CR']:+.2%}")